In [47]:
!python --version
# !pip list > requirements.txt
!echo "---------------------------------------------"
!cat requirements.txt | grep "torch"

Python 3.10.12
---------------------------------------------
torch                            2.2.1+cu121
torchaudio                       2.2.1+cu121
torchdata                        0.7.1
torchsummary                     1.5.1
torchtext                        0.17.1
torchvision                      0.17.1+cu121


# Deep Residual Learning for Image Recognition

논문URL - https://arxiv.org/abs/1512.03385v1


## Residual Block

<img src="https://d3i71xaburhd42.cloudfront.net/2c03df8b48bf3fa39054345bafabfeff15bfd11d/2-Figure2-1.png" alt="Figure 2" />

### Basic Block & Bottleneck Block
컨볼루션 잔여 블록의 두 가지 변형.
- 왼쪽: 3x3 컨볼루션 레이어 2개가 있는 기본 블록.
- 오른쪽: 차원 감소(예: 1/4)를 위한 1x1 컨볼루션 레이어, 3x3 컨볼루션 레이어, 차원 복원을 위한 또 다른 1x1 컨볼루션 레이어가 있는 병목 블록입니다.

<img src="https://d3i71xaburhd42.cloudfront.net/2c03df8b48bf3fa39054345bafabfeff15bfd11d/6-Figure5-1.png" alt="Figure 5"/>

In [48]:
import torch
import torch.nn as nn

In [49]:
dim = 64
input = torch.rand(1, 64, 56, 56)

# Basic Block
layers = nn.Sequential(
    nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(dim),
    nn.ReLU(),
    nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(dim)
)

out = layers(input) + input
out = torch.relu(out)
print(f"input shape: {input.shape}")
print(f"output shape: {out.shape}")

input shape: torch.Size([1, 64, 56, 56])
output shape: torch.Size([1, 64, 56, 56])


In [50]:
dim = 256
input = torch.rand(1, 256, 56, 56)

# Bottleneck Block
layers = nn.Sequential(
    nn.Conv2d(dim, dim//4, kernel_size=1, stride=1),
    nn.BatchNorm2d(dim//4),
    nn.ReLU(),
    nn.Conv2d(dim//4, dim//4, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(dim//4),
    nn.ReLU(),
    nn.Conv2d(dim//4, dim, kernel_size=1, stride=1),
    nn.BatchNorm2d(dim)
)

out = layers(input) + input
out = torch.relu(out)
print(f"input shape: {input.shape}")
print(f"output shape: {out.shape}")

input shape: torch.Size([1, 256, 56, 56])
output shape: torch.Size([1, 256, 56, 56])


### Residual Block with downsampling

In [51]:
dim = 64
input = torch.rand(1, 64, 56, 56)

# Basic Block
layers = nn.Sequential(
    nn.Conv2d(dim, dim, kernel_size=3, stride=2, padding=1), # stride 2
    nn.BatchNorm2d(dim),
    nn.ReLU(),
    nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(dim)
)

downsample = nn.Sequential(
    nn.Conv2d(dim, dim, kernel_size=1, stride=2),
    nn.BatchNorm2d(dim)
)

out = layers(input) + downsample(input)
out = torch.relu(out)
print(f"input shape: {input.shape}")
print(f"output shape: {out.shape}")

input shape: torch.Size([1, 64, 56, 56])
output shape: torch.Size([1, 64, 28, 28])


In [52]:
dim = 256
input = torch.rand(1, 256, 56, 56)

# Bottleneck Block
layers = nn.Sequential(
    nn.Conv2d(dim, dim//4, kernel_size=1, stride=2), # stride 2
    nn.BatchNorm2d(dim//4),
    nn.ReLU(),
    nn.Conv2d(dim//4, dim//4, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(dim//4),
    nn.ReLU(),
    nn.Conv2d(dim//4, dim, kernel_size=1, stride=1),
    nn.BatchNorm2d(dim)
)

downsample = nn.Sequential(
    nn.Conv2d(dim, dim, kernel_size=1, stride=2),
    nn.BatchNorm2d(dim)
)

out = layers(input) + downsample(input)
out = torch.relu(out)
print(f"input shape: {input.shape}")
print(f"output shape: {out.shape}")

input shape: torch.Size([1, 256, 56, 56])
output shape: torch.Size([1, 256, 28, 28])


## Resnet Architecture

<img src="https://d3i71xaburhd42.cloudfront.net/2c03df8b48bf3fa39054345bafabfeff15bfd11d/5-Table1-1.png" alt="Table1" />

## Resnet 구현

- conv1:
    - input size: 224 x 224
    - in_channels: 3(RGB)
    - out_channels: 64
    - kernel_size: 7
    - stride: 2
    - padding: 3(calculated)
        - output size: 112 x 112

In [53]:
input = torch.rand(1, 3, 224, 224)

conv1 = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm2d(64)
)

out = conv1(input)
print(f"input shape: {input.shape}")
print(f"output shape: {out.shape}")

input shape: torch.Size([1, 3, 224, 224])
output shape: torch.Size([1, 64, 112, 112])


- max_pool:
    - input size: 112 x 112
    - kernel_size: 3
    - stride: 2
    - padding: 1(calculated)
        - output size: 56 x 56

In [54]:
input = torch.rand(1, 64, 112, 112)
max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

out = max_pool(input)
print(f"input shape: {input.shape}")
print(f"output shape: {out.shape}")

input shape: torch.Size([1, 64, 112, 112])
output shape: torch.Size([1, 64, 56, 56])


- Basic Block(18, 34 layer)
    - input size: 56, 28, 14, 7
    - kernel_size: 3
    - stride: 1 or 2
    - padding: 1
        - output size: 56, 28, 14, 7

- Downsample 적용(ex. 18-layer)
    - Basic Block 은 stride가 2인 conv3_1, conv4_1, conv5_1에서 feature map size 가 1/2 축소됨
    - input x에 downsample을 적용한 후 output에 더해 잔차 학습을 진행


|   conv   | in_dim | dim | out_dim | stride | projection |
| :------: | :----: | :-: | :----:  |:-----: | :--------: |
| conv2_1  |   64   | 64  |   64    |  1     |     X      |
| conv2_2  |   64   | 64  |   64    |  1     |     X      |
| conv3_1  |   64   | 128 |   128   |  2     |     O      |
| conv3_2  |  128   | 128 |   128   |  1     |     X      |
| conv4_1  |  128   | 256 |   256   |  2     |     O      |
| conv4_2  |  256   | 256 |   256   |  1     |     X      |
| conv5_1  |  256   | 512 |   256   |  2     |     O      |
| conv5_2  |  512   | 512 |   256   |  1     |     X      |


In [55]:
# class BasicBlock(nn.Module):
#     def __init__(self, in_dim, dim, stride=1):
#         super().__init__()
#         self.conv1 = nn.Conv2d(in_dim, dim, kernel_size=3, stride=stride, padding=1)
#         self.bn = nn.BatchNorm2d(dim)
#         self.relu = nn.ReLU()
#         self.conv2 = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1)

#     def forward(self, x):
#         input = x
#         out = self.conv1(x)
#         out = self.bn(out)
#         out = self.relu(out)
#         out = self.conv2(out)
#         out = self.bn(out)

#         out = out + input
#         out = self.relu(out)
#         return out

In [56]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_dim, dim, stride=1):
        super().__init__()
        self.stride = stride
        self.conv1 = nn.Conv2d(in_dim, dim, kernel_size=3, stride=stride, padding=1)
        self.bn = nn.BatchNorm2d(dim)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1)
        self.downsample = self._build_downsample_layer(in_dim, dim, stride)

    def forward(self, x):
        input = x
        out = self.conv1(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn(out)

        if self.stride != 1:
            input = self.downsample(input)

        out = out + input
        out = self.relu(out)
        return out

    def _build_downsample_layer(self, in_dim, dim, stride):
        return nn.Sequential(
            nn.Conv2d(in_dim, dim * self.expansion, kernel_size=1, stride=stride),
            nn.BatchNorm2d(dim * self.expansion)
        )

In [57]:
input = torch.rand(1, 64, 56, 56)
basic_block = BasicBlock(64, 64, 1)
out = basic_block(input)
print(f"input shape: {input.shape}")
print(f"output shape: {out.shape}")

input shape: torch.Size([1, 64, 56, 56])
output shape: torch.Size([1, 64, 56, 56])


In [58]:
input = torch.rand(1, 64, 56, 56)
basic_block = BasicBlock(64, 128, 2)
out = basic_block(input)
print(f"input shape: {input.shape}")
print(f"output shape: {out.shape}")

input shape: torch.Size([1, 64, 56, 56])
output shape: torch.Size([1, 128, 28, 28])


- Bottleneck Block(18, 34 layer)
    - input size: 56, 28, 14, 7
    - kernel_size: 1 or 3
    - stride: 1 or 2
    - padding: 0 or 1
        - output size: 56, 28, 14, 7

- Downsample 적용(ex. 50-layer)
    - Bottleneck Block 은 stride가 2(feature map size 축소)인 conv3_1, conv4_1, conv5_1와 channel 개수가 변하는 conv2_1에서 donwsample 적용

|   conv   | in_dim | dim | out_dim | stride | projection |
| :------: | :----: | :-: | :-----: | :----: | :--------: |
| conv2_1  |   64   | 64  |   256   |   1    |     O      |
| conv2_2  |  256   | 64  |   256   |   1    |     X      |
| conv2_3  |  256   | 64  |   256   |   1    |     X      |
| conv3_1  |  256   | 128 |   512   |   2    |     O      |
| conv3_2  |  512   | 128 |   512   |   1    |     X      |
| conv3_3  |  512   | 128 |   512   |   1    |     X      |
| conv3_4  |  512   | 128 |   512   |   1    |     X      |
| conv4_1  |  512   | 256 |   1024  |   2    |     O      |
| conv4_2  |  1024  | 256 |   1024  |   1    |     X      |
| conv4_3  |  1024  | 256 |   1024  |   1    |     X      |
| conv4_4  |  1024  | 256 |   1024  |   1    |     X      |
| conv4_5  |  1024  | 256 |   1024  |   1    |     X      |
| conv4_6  |  1024  | 256 |   1024  |   1    |     X      |
| conv5_1  |  1024  | 512 |   2048  |   2    |     O      |
| conv5_2  |  2048  | 512 |   2048  |   1    |     X      |
| conv5_3  |  2048  | 512 |   2048  |   1    |     X      |
| conv5_4  |  2048  | 512 |   2048  |   1    |     X      |



In [59]:
# class BottleneckBlock(nn.Module):
#     def __init__(self, in_dim, dim, stride=1):
#         super().__init__()
#         self.stride = stride
#         self.conv1 = nn.Conv2d(in_dim, dim, kernel_size=1, stride=stride, padding=0)
#         self.bn1 = nn.BatchNorm2d(dim)
#         self.relu = nn.ReLU()
#         self.conv2 = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1)
#         self.bn2 = nn.BatchNorm2d(dim)
#         self.conv3 = nn.Conv2d(dim, dim * 4, kernel_size=1, stride=1, padding=0)
#         self.bn3 = nn.BatchNorm2d(dim * 4)
#         self.downsample = self._build_downsample_layer(in_dim, dim, stride)

#     def forward(self, x):
#         input = x
#         out = self.conv1(x)
#         out = self.bn1(out)
#         out = self.relu(out)
#         out = self.conv2(out)
#         out = self.bn2(out)
#         out = self.relu(out)
#         out = self.conv3(out)
#         out = self.bn3(out)

#         if self.stride != 1:
#             input = self.downsample(input)

#         out = out + input
#         out = self.relu(out)
#         return out

#     def _build_downsample_layer(self, in_dim, dim, stride):
#         return nn.Sequential(
#             nn.Conv2d(in_dim, dim * 4, kernel_size=1, stride=stride),
#             nn.BatchNorm2d(dim * 4)
#         )

In [60]:
class BottleneckBlock(nn.Module):
    expansion = 4

    def __init__(self, in_dim, dim, stride=1):
        super().__init__()
        self.in_dim = in_dim
        self.dim = dim
        self.stride = stride
        self.conv1 = nn.Conv2d(in_dim, dim, kernel_size=1, stride=stride, padding=0)
        self.bn1 = nn.BatchNorm2d(dim)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(dim)
        self.conv3 = nn.Conv2d(dim, dim * self.expansion, kernel_size=1, stride=1, padding=0)
        self.bn3 = nn.BatchNorm2d(dim * self.expansion)
        self.downsample = self._build_downsample_layer(in_dim, dim, stride)

    def forward(self, x):
        input = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv3(out)
        out = self.bn3(out)

        if self.stride != 1 or self.in_dim != self.dim * self.expansion:
            input = self.downsample(input)

        out = out + input
        out = self.relu(out)
        return out

    def _build_downsample_layer(self, in_dim, dim, stride):
        return nn.Sequential(
            nn.Conv2d(in_dim, dim * self.expansion, kernel_size=1, stride=stride),
            nn.BatchNorm2d(dim * self.expansion)
        )

In [61]:
input = torch.rand(1, 64, 56, 56)

bottleneck_block = BottleneckBlock(64, 64, 1)
out = bottleneck_block(input)
print(f"input shape: {input.shape}")
print(f"output shape: {out.shape}")

input shape: torch.Size([1, 64, 56, 56])
output shape: torch.Size([1, 256, 56, 56])


In [62]:
input = torch.rand(1, 256, 56, 56)

bottleneck_block = BottleneckBlock(256, 128, 2)
out = bottleneck_block(input)
print(f"input shape: {input.shape}")
print(f"output shape: {out.shape}")

input shape: torch.Size([1, 256, 56, 56])
output shape: torch.Size([1, 512, 28, 28])


In [63]:
IMAGE_SIZE = 224

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes):
        super().__init__()
        self.in_dim = 64
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64)
        )
        self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.stage1 = self._build_stage(block, 64, layers[0], stride=1)
        self.stage2 = self._build_stage(block, 128, layers[1], stride=2)
        self.stage3 = self._build_stage(block, 256, layers[2], stride=2)
        self.stage4 = self._build_stage(block, 512, layers[3], stride=2)

        self.average_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for layers in [self.stem, self.stage1, self.stage2, self.stage3, self.stage4, self.fc]:
            if isinstance(layers, nn.Linear):
                nn.init.xavier_uniform_(layers.weight)
                continue

            for layer in layers:
                if isinstance(layer, nn.Conv2d):
                    nn.init.xavier_uniform_(layer.weight)

    def _build_stage(self, block, dim, layers, stride):
        stage = []
        stage.append(block(self.in_dim, dim, stride))
        self.in_dim = dim * block.expansion

        for _ in range(layers - 1):
            stage.append(block(self.in_dim, dim, stride=1))

        return nn.Sequential(*stage)

    def forward(self, x):
        out = self.stem(x)
        out = self.max_pool(out)
        out = self.stage1(out)
        out = self.stage2(out)
        out = self.stage3(out)
        out = self.stage4(out)
        out = self.average_pool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

In [64]:
resnet18 = ResNet(BasicBlock, [2, 2, 2, 2], 1000)
x = torch.rand(1, 3, 224, 224)
out = resnet18(x)
out.shape

torch.Size([1, 1000])

In [65]:
try:
    from torchinfo import summary
except Exception as ex:
    !pip install torchinfo
    from torchinfo import summary

In [66]:
summary(resnet18, [1, 3, 224, 224])

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Sequential: 1-1                        [1, 64, 112, 112]         --
│    └─Conv2d: 2-1                       [1, 64, 112, 112]         9,472
│    └─BatchNorm2d: 2-2                  [1, 64, 112, 112]         128
├─MaxPool2d: 1-2                         [1, 64, 56, 56]           --
├─Sequential: 1-3                        [1, 64, 56, 56]           --
│    └─BasicBlock: 2-3                   [1, 64, 56, 56]           4,288
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           36,928
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           36,928
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]           (recursive)
│    │    └─ReLU: 3-6                    [1, 64, 56, 56]    

In [67]:
import torch

In [68]:
num_classes = 1000

resnet50 = ResNet(BottleneckBlock, [3, 4, 6, 3], num_classes)
x = torch.rand(1, 3, 224, 224)
out = resnet50(x)
assert out.shape == torch.Size([1, num_classes])

In [69]:
summary(resnet50, [1, 3, 224, 224])

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Sequential: 1-1                        [1, 64, 112, 112]         --
│    └─Conv2d: 2-1                       [1, 64, 112, 112]         9,472
│    └─BatchNorm2d: 2-2                  [1, 64, 112, 112]         128
├─MaxPool2d: 1-2                         [1, 64, 56, 56]           --
├─Sequential: 1-3                        [1, 256, 56, 56]          --
│    └─BottleneckBlock: 2-3              [1, 256, 56, 56]          --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           4,160
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           36,928
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-6                    [1, 64, 56, 56]           --
│ 

In [70]:
def resnet18(num_classes):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

def resnet34(num_classes):
    return ResNet(BasicBlock, [3, 4, 6, 3], num_classes)

def resnet50(num_classes):
    return ResNet(BottleneckBlock, [3, 4, 6, 3], num_classes)

def resnet101(num_classes):
    return ResNet(BottleneckBlock, [3, 4, 23, 3], num_classes)

def resnet152(num_classes):
    return ResNet(BottleneckBlock, [3, 8, 36, 3], num_classes)

## Cifar-10 데이터 학습

In [71]:
# download cifar-10 dataset
!wget -O ./cifar-10-python.tar.gz https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
!tar -xf ./cifar-10-python.tar.gz

--2024-05-08 01:54:18--  https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
Resolving www.cs.toronto.edu (www.cs.toronto.edu)... 128.100.3.30
Connecting to www.cs.toronto.edu (www.cs.toronto.edu)|128.100.3.30|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 170498071 (163M) [application/x-gzip]
Saving to: ‘./cifar-10-python.tar.gz’

./cifar-10-python.t 100%[===================>] 162.60M  37.3MB/s    in 4.9s    

2024-05-08 01:54:23 (33.5 MB/s) - ‘./cifar-10-python.tar.gz’ saved [170498071/170498071]



In [72]:
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

In [73]:
class CifarDataset(Dataset):
    def __init__(self, cifar_path, transform=None, train=True):
        filename = "data_batch_*" if train else "test_batch"
        batch_files = list(Path(cifar_path).glob(filename))

        self.images = []
        self.labels = []
        self.transform = transform

        for file in batch_files:
            batch = self.unpickle(str(file))
            self.labels += batch[b'labels']
            data = batch[b'data']

            for image in data:
                image = [np.split(x, 32) for x in np.split(image, 3)]
                image = np.array(image)
                self.images.append(image)

        self.images = torch.from_numpy(np.array(self.images))

    def unpickle(self, file):
        import pickle
        with open(file, 'rb') as fo:
            dict = pickle.load(fo, encoding='bytes')
        return dict

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [74]:
transform = v2.Compose([
    v2.RandomChoice([v2.Resize(256), v2.Resize(480)]),
    v2.Resize((224, 224)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [75]:
train_dataset = CifarDataset("./cifar-10-batches-py", transform=transform, train=True)
test_dataset = CifarDataset("./cifar-10-batches-py", transform=transform, train=False)

In [76]:
assert len(train_dataset) == 50_000
assert len(test_dataset) == 10_000

In [77]:
from torch.utils.data import random_split

train_dataset, validation_dataset = random_split(train_dataset, [0.8, 0.2])

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
validation_dataloader = DataLoader(validation_dataset, batch_size=16, shuffle=False)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

epochs = 20
num_classes = 10
lr = 1e-5

model = resnet50(num_classes).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr)

train_loss_hist, test_loss_hist = [], []
train_accuracy_hist, test_accuracy_hist = [], []

for epoch in range(epochs):
    train_loss, test_loss = 0, 0
    train_accuracy, test_accuracy = 0, 0

    model.train()
    for X_train, y_train in train_dataloader:
        X_train, y_train = X_train.to(device), y_train.to(device)

        pred_logits = model(X_train)
        preds = torch.argmax(pred_logits, dim=1)
        train_accuracy += (preds == y_train).sum().item()
        loss = loss_fn(pred_logits, y_train)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        for X_test, y_test in validation_dataloader:
            X_test, y_test = X_test.to(device), y_test.to(device)

            pred_logits = model(X_test)
            preds = torch.argmax(pred_logits, dim=1)
            test_accuracy += (preds == y_test).sum().item()
            loss = loss_fn(pred_logits, y_test)
            test_loss += loss.item()

    train_loss /= len(train_dataset)
    test_loss /= len(test_dataset)
    train_accuracy /= len(train_dataset)
    test_accuracy /= len(test_dataset)

    train_loss_hist.append(train_loss)
    test_loss_hist.append(test_loss)
    train_accuracy_hist.append(train_accuracy)
    test_accuracy_hist.append(test_accuracy)

    print(f"epoch: {epoch+1} | train loss: {train_loss:.4f} | train accuracy: {train_accuracy * 100:.4f} | test loss: {test_loss:.4f} test accuracy: {test_accuracy * 100:.4f}")

epoch: 1 | train loss: 0.1147 | train accuracy: 32.6375 | test loss: 0.1072 test accuracy: 39.9800
epoch: 2 | train loss: 0.0970 | train accuracy: 43.3300 | test loss: 0.0932 test accuracy: 48.0100
epoch: 3 | train loss: 0.0879 | train accuracy: 48.9850 | test loss: 0.0822 test accuracy: 52.6300
epoch: 4 | train loss: 0.0800 | train accuracy: 53.7975 | test loss: 0.0756 test accuracy: 57.4400
epoch: 5 | train loss: 0.0722 | train accuracy: 58.7300 | test loss: 0.0664 test accuracy: 62.5200
epoch: 6 | train loss: 0.0650 | train accuracy: 63.1350 | test loss: 0.0609 test accuracy: 66.2400
epoch: 7 | train loss: 0.0586 | train accuracy: 67.0275 | test loss: 0.0555 test accuracy: 69.1000
epoch: 8 | train loss: 0.0524 | train accuracy: 70.5225 | test loss: 0.0513 test accuracy: 72.1900
epoch: 9 | train loss: 0.0473 | train accuracy: 73.5125 | test loss: 0.0493 test accuracy: 72.8100
epoch: 10 | train loss: 0.0431 | train accuracy: 76.0825 | test loss: 0.0446 test accuracy: 75.1700
epoch: 11